> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# IFR2 on liquid names: one fixed setting, Ibovespa names and the most traded BDRs

`ifr2_universe.ipynb` asked which tickers work with a setup. This notebook narrows the question the owner raised after it:

> With **one fixed IFR2 setting** (no per-ticker tuning), restricted to **liquid Ibovespa names** and to the **ten most traded BDRs**, does anything beat the CDI and the IBOV out of sample?

The rules live in `docs/brief.md` section 7, the score in `docs/score.md`, the benchmarks in `docs/cdi.md` and `docs/ibov.md`. The notebook holds no trading logic: it calls `b3core` and prints tables.

**Before trusting any number here:**
- **The fixed setting was not chosen blind.** RSI entry 20 and target lookback 2 came from exploratory runs on the same history, so the *in-sample* columns are optimistic. Only the out-of-sample columns (a date no choice used) deserve weight, and even they come from one regime.
- **The ticker lists are drafts.** They are written from memory, not taken from B3's current composition. Tickers with no data are listed, never silently dropped. Edit the lists freely.
- **"Liquid" and "best" use data.** Liquidity is the median traded value (close times volume) over the last year of each ticker's candles, so the list favours names that survived and stayed liquid. That is survivorship bias, and it flatters a mean-reversion strategy.
- **Picking the best of many is the same trap as picking the best parameters.** A set chosen in-sample must be judged out-of-sample against random sets of the same size (section 6).
- **BDRs carry the dollar.** A BDR's price moves with the US stock *and* the USD/BRL rate, and its Yahoo history is often thin, with flat days when nobody traded. The IBOV and the CDI are not like-for-like benchmarks for it.
- **Not modelled:** taxes, slippage beyond the 0.03% fee, dividends (raw closes), and a shared-cash rule for simultaneous signals. The portfolio is independent slots, as in `docs/` and `b3core.portfolio`.
- **Not here on purpose:** the "same-days" comparison (the CDI over only the days in a trade) and a yearly walk-forward pick. Both were explored in scratch code, and neither has a spec or a test yet (CLAUDE.md rule 2).

In [ ]:
import json
import urllib.request
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

from dataclasses import replace

from b3core.benchmark import compare_with_index
from b3core.ifr2 import Ifr2Config
from b3core.portfolio import buy_and_hold_equity, combine_equity, slot_equity, summarize_portfolio
from b3core.universe import scan_universe

pd.options.display.float_format = "{:,.4f}".format
CACHE = Path("../samples/ifr2_universe")  # gitignored vendor data, shared with ifr2_universe.ipynb
CACHE.mkdir(parents=True, exist_ok=True)

## 1. Parameters

`CONFIG` is the one fixed setting. Everything else is a list or a size you can change.

In [ ]:
CONFIG = Ifr2Config.preset(
    rsi_entry=20, exit_mode="target", target_lookback=2, max_days=None,
    atr_period=5, atr_mult=2, max_risk_brl=2_000, capital_brl=100_000, lot_size=1,
)
START = date(2012, 1, 1)
END = date(2027, 1, 1)  # exclusive
OOS_FRACTION = 0.3

# Draft candidates, from memory: edit freely. Tickers that fail to download are reported below.
IBOV_CANDIDATES = [
    "PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "ABEV3", "WEGE3", "RENT3", "LREN3", "MGLU3",
    "SUZB3", "GGBR4", "CSNA3", "USIM5", "JBSS3", "RADL3", "RAIL3", "EQTL3", "CMIG4", "ELET3",
    "TOTS3", "PRIO3", "CSAN3", "BBSE3", "SBSP3", "VIVT3", "B3SA3", "BRAP4", "GOAU4", "CYRE3",
    "ITSA4", "KLBN11", "TAEE11", "SANB11", "BPAC11", "HAPV3", "COGN3", "BEEF3", "MRFG3", "CPLE6",
    "EGIE3", "SMTO3", "RDOR3", "HYPE3", "CCRO3", "UGPA3", "VBBR3", "EMBR3", "ALPA4", "MULT3",
    "BRFS3", "ASAI3", "GMAT3", "CMIN3", "ENEV3", "RRRP3",
]
BDR_CANDIDATES = [
    "AAPL34", "MSFT34", "AMZO34", "GOGL34", "NVDC34", "TSLA34", "M1TA34", "NFLX34",
    "DISB34", "COCA34", "JPMC34", "BERK34", "MUTC34", "ROXO34", "MELI34", "BABA34",
]
N_LIQUID_IBOV = 30  # the Ibovespa universe: the most traded of the candidates
N_LIQUID_BDR = 10  # the BDR universe: the ten most traded of the candidates
PICK_SIZES = [5, 8, 10]  # how many Ibovespa names to keep when choosing the best in-sample
MIN_CANDLES_IBOV = 750  # about three years of daily candles
MIN_CANDLES_BDR = 500  # BDRs are younger: about two years
LIQUIDITY_WINDOW = 250  # candles used to measure the median traded value

## 2. Load candles

Daily candles from Yahoo Finance, raw prices, one CSV per ticker in the shared cache folder (the files `ifr2_universe.ipynb` already downloaded are reused). A **data-quality table** follows, because thin history and flat days matter most for BDRs.

In [ ]:
def load_candles(ticker: str) -> pd.DataFrame | None:
    path = CACHE / f"{ticker}.csv"
    if not path.exists():
        raw = yf.download(f"{ticker}.SA", start=START, end=END, auto_adjust=False, progress=False)
        if raw.empty:
            return None
        raw.columns = raw.columns.get_level_values(0)
        raw = raw.dropna(subset=["Open", "High", "Low", "Close"])
        raw[["Open", "High", "Low", "Close", "Volume"]].to_csv(path)
    raw = pd.read_csv(path, index_col=0, parse_dates=True)
    return pd.DataFrame(
        {
            "ts": raw.index.tz_localize("America/Sao_Paulo").tz_convert("UTC"),
            "open": raw["Open"].to_numpy(),
            "high": raw["High"].to_numpy(),
            "low": raw["Low"].to_numpy(),
            "close": raw["Close"].to_numpy(),
            "volume": raw["Volume"].to_numpy(dtype=float),
        }
    ).reset_index(drop=True)


def load_ibov() -> pd.Series:
    path = CACHE / "ibov.csv"
    if not path.exists():
        raw = yf.download("^BVSP", start=START, end=END, auto_adjust=False, progress=False)
        raw.columns = raw.columns.get_level_values(0)
        raw[["Close"]].dropna().to_csv(path)
    return pd.read_csv(path, index_col=0, parse_dates=True)["Close"].dropna()


def local_day(candles: pd.DataFrame) -> pd.DatetimeIndex:
    return pd.DatetimeIndex(candles["ts"]).tz_convert("America/Sao_Paulo").tz_localize(None).normalize()


ibov = load_ibov()


def load_universe(tickers: list[str], min_candles: int) -> tuple[dict, dict]:
    """Raw candles per ticker, and the tickers left out (no data, or too little history)."""
    loaded = {t: load_candles(t) for t in tickers}
    out = {
        "no data": [t for t, c in loaded.items() if c is None],
        "too short": [t for t, c in loaded.items() if c is not None and len(c) < min_candles],
    }
    data = {t: c for t, c in loaded.items() if c is not None and len(c) >= min_candles}
    return data, out


ibov_all, ibov_out = load_universe(IBOV_CANDIDATES, MIN_CANDLES_IBOV)
bdr_all, bdr_out = load_universe(BDR_CANDIDATES, MIN_CANDLES_BDR)
print(f"Ibovespa candidates: {len(ibov_all)} loaded; left out: {ibov_out}")
print(f"BDR candidates: {len(bdr_all)} loaded; left out: {bdr_out}")
print(f"IBOV: {len(ibov)} closes, {ibov.index[0].date()} to {ibov.index[-1].date()}")

In [ ]:
def quality(data: dict) -> pd.DataFrame:
    """Plain data checks: history, zero-volume days, flat days (high equals low), the longest gap."""
    rows = {}
    for t, c in data.items():
        days = local_day(c)
        rows[t] = {
            "candles": len(c), "first": days[0].date(), "last": days[-1].date(),
            "zero volume %": float((c["volume"] == 0).mean() * 100),
            "flat days %": float((c["high"] == c["low"]).mean() * 100),
            "longest gap (days)": int(pd.Series(days).diff().dt.days.max()),
        }
    return pd.DataFrame(rows).T


quality(bdr_all)

## 3. The CDI

The benchmark is the daily CDI compounded over each window (`docs/cdi.md`). The Central Bank API is tried first; if it is blocked, the **Selic target minus 0.10 points a year** is used, which is an approximation (the cell prints which one). A file `samples/ifr2_universe/cdi.csv` (series 12, columns `date` and `rate`) overrides both.

In [ ]:
def fetch(url: str) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8")


trading_days = sorted(set().union(*[set(local_day(c)) for d in (ibov_all, bdr_all) for c in d.values()]))


def cdi_from_bcb() -> pd.Series:
    rows, begin = [], START
    while begin < END and begin < datetime.now(UTC).date():
        stop = min(begin + timedelta(days=3_000), END - timedelta(days=1))  # about 8 years
        url = (
            "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
            f"?formato=json&dataInicial={begin:%d/%m/%Y}&dataFinal={stop:%d/%m/%Y}"
        )
        rows += json.loads(fetch(url))  # raises when the API answers with an HTML error page
        begin = stop + timedelta(days=1)
    index = pd.to_datetime([r["data"] for r in rows], format="%d/%m/%Y")
    return pd.Series([float(r["valor"]) for r in rows], index=index).sort_index()


def cdi_from_selic_target() -> pd.Series:
    url = "http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='BM366_TJOVER366')"
    values = json.loads(fetch(url))["value"]
    annual = pd.Series(
        [v["VALVALOR"] for v in values], index=pd.to_datetime([v["VALDATA"][:10] for v in values])
    )
    daily = (1 + (annual - 0.10) / 100) ** (1 / 252) - 1  # percent a year to a business-day rate
    return (daily * 100).reindex(trading_days).dropna()  # trading days only


def load_cdi() -> tuple[pd.Series, str]:
    exact = CACHE / "cdi.csv"
    approx = CACHE / "cdi_from_selic_target.csv"
    if not exact.exists() and not approx.exists():
        try:
            cdi_from_bcb().rename_axis("date").rename("rate").to_csv(exact, date_format="%d/%m/%Y")
        except Exception as error:  # the API blocks some networks
            print(f"Central Bank API unavailable ({type(error).__name__}); using the Selic target")
            cdi_from_selic_target().rename_axis("date").rename("rate").to_csv(approx, date_format="%d/%m/%Y")
    path = exact if exact.exists() else approx
    frame = pd.read_csv(path)
    index = pd.to_datetime(frame["date"], format="%d/%m/%Y")
    series = pd.Series(frame["rate"].to_numpy(dtype=float), index=index).sort_index()
    label = "CDI (BCB series 12)" if path == exact else "Selic target minus 0.10 points (approximation)"
    return series, label


cdi, CDI_SOURCE = load_cdi()
print(f"source: {CDI_SOURCE}")
print(f"{len(cdi)} days, {cdi.index[0].date()} to {cdi.index[-1].date()}")

## 4. Clean the candles, then build the two universes

Two kinds of row cannot be compared with the CDI, and they are **dropped and printed**, never filled:
- **Candles before the first IBOV close** (`docs/ibov.md`): Yahoo's index series starts a day after the first stock candle.
- **Candles on days with no CDI rate.** With the exact CDI (series 12) these are holidays that Yahoo pads with a zero-volume row, and the days after the latest published rate. With the Selic-target fallback every weekday has a rate, so this check drops nothing.

**Most traded** is then the median of `close * volume` over the last `LIQUIDITY_WINDOW` candles of each ticker. It is measured on the most recent data, so it is a survivor ranking, not what was liquid in 2013.

In [ ]:
def clean(data: dict) -> tuple[dict, pd.DataFrame]:
    cleaned, rows = {}, {}
    cdi_days = set(pd.DatetimeIndex(cdi.index).normalize())
    for t, c in data.items():
        days = local_day(c)
        before_ibov = days < ibov.index[0]
        no_rate = ~before_ibov & ~days.isin(cdi_days) & (days > ibov.index[0])
        zero_volume = no_rate & (c["volume"].to_numpy() == 0)
        keep = ~before_ibov & ~no_rate
        cleaned[t] = c[keep].reset_index(drop=True)
        rows[t] = {
            "before first IBOV close": int(before_ibov.sum()),
            "no CDI rate": int(no_rate.sum()),
            "...of which zero volume": int(zero_volume.sum()),
        }
    return cleaned, pd.DataFrame(rows).T


def traded_value(data: dict) -> pd.Series:
    return pd.Series(
        {t: float((c["close"] * c["volume"]).tail(LIQUIDITY_WINDOW).median()) for t, c in data.items()}
    ).sort_values(ascending=False)


ibov_clean, ibov_dropped = clean(ibov_all)
bdr_clean, bdr_dropped = clean(bdr_all)
dropped = pd.concat([ibov_dropped, bdr_dropped])
print(f"{int((dropped > 0).any(axis=1).sum())} of {len(dropped)} tickers had rows dropped. Most dropped for one ticker:")
print(dropped.max().to_string())

ibov_liquidity = traded_value(ibov_clean)
bdr_liquidity = traded_value(bdr_clean)
ibov_data = {t: ibov_clean[t] for t in ibov_liquidity.index[:N_LIQUID_IBOV]}
bdr_data = {t: bdr_clean[t] for t in bdr_liquidity.index[:N_LIQUID_BDR]}
print(f"Ibovespa universe ({len(ibov_data)}): {sorted(ibov_data)}")
print(f"BDR universe ({len(bdr_data)}): {sorted(bdr_data)}")
pd.DataFrame({"ibov: median traded value (R$)": ibov_liquidity, "bdr: median traded value (R$)": bdr_liquidity})

## 5. One shared out-of-sample date per universe

Each universe is cut at the date that leaves roughly the last `OOS_FRACTION` of its trading days out. Everything that is *chosen* (which Ibovespa names to keep) is chosen on the in-sample part only.

In [ ]:
def split(data: dict) -> tuple[dict, dict, pd.Timestamp]:
    days = {t: local_day(c) for t, c in data.items()}
    all_days = sorted(set().union(*days.values()))
    cut = all_days[int(len(all_days) * (1 - OOS_FRACTION))]
    return (
        {t: c[days[t] < cut].reset_index(drop=True) for t, c in data.items()},
        {t: c[days[t] >= cut].reset_index(drop=True) for t, c in data.items()},
        cut,
    )


ibov_is, ibov_oos, IBOV_CUT = split(ibov_data)
bdr_is, bdr_oos, BDR_CUT = split(bdr_data)
print(f"Ibovespa universe: out-of-sample starts {IBOV_CUT.date()}")
print(f"BDR universe: out-of-sample starts {BDR_CUT.date()}")

## 6. Ibovespa names: the same setting on every ticker, then the best in-sample

The fixed `CONFIG` runs on each ticker, in-sample and out-of-sample side by side. `excess` is the strategy's CAGR minus the CDI's over the same window and `beats_cdi` is true when it is positive. Read `max_equity_drawdown` and `calmar` next to it. Ranked by the in-sample `score_v1`.

In [ ]:
SCAN_COLUMNS = ["n_trades", "result", "cagr", "excess", "beats_cdi", "max_equity_drawdown", "calmar", "score_v1",
                "bh_cagr", "beats_bh", "ibov_cagr", "beats_ibov"]


def scan_both(in_sample: dict, out_of_sample: dict, min_is: int, min_oos: int) -> pd.DataFrame:
    scan_is = scan_universe(in_sample, CONFIG, cdi, min_candles=min_is, index_closes=ibov)
    scan_oos = scan_universe(out_of_sample, CONFIG, cdi, min_candles=min_oos, index_closes=ibov)
    print("skipped in-sample:", scan_is.attrs["skipped"] or "none", "| out-of-sample:", scan_oos.attrs["skipped"] or "none")
    table = scan_is[SCAN_COLUMNS].add_prefix("is_").join(scan_oos[SCAN_COLUMNS].add_prefix("oos_"), how="left")
    return table.sort_values("is_score_v1", ascending=False)


scan_ibov = scan_both(ibov_is, ibov_oos, MIN_CANDLES_IBOV // 2, 120)
scan_ibov

In [ ]:
both = scan_ibov.dropna(subset=["oos_beats_cdi"])
pd.Series(
    {
        "tickers": len(scan_ibov),
        "beat the CDI in-sample": int(scan_ibov["is_beats_cdi"].sum()),
        "beat the CDI out-of-sample": int(both["oos_beats_cdi"].sum()),
        "beat it in both": int((both["is_beats_cdi"] & both["oos_beats_cdi"]).sum()),
        "median in-sample excess": scan_ibov["is_excess"].median(),
        "median out-of-sample excess": both["oos_excess"].median(),
    },
    name="summary",
).to_frame()

### 6b. Keep the best N in-sample, judge them out-of-sample

The tickers are ranked by their in-sample `score_v1`, the best N form an equal-split portfolio (`b3core.portfolio`, independent slots of `capital / N`), and the portfolio is judged on the out-of-sample part. The **worst N** are the control: if the ranking means anything they should do worse. `beats % of random` is the share of random N-ticker portfolios from the same universe with a lower out-of-sample CAGR: near 50% means the choice is no better than chance, near 100% means it persisted.

Two risk levels: the same total risk budget (x1) and three times it (x3). With only a few names a slot risks little, so the portfolio is often idle; x3 shows what deploying more capital does to return *and* drawdown.

In [ ]:
CAPITAL = CONFIG.capital_brl
DRAWS = 300  # random subsets per size
RISK_SCALES = {"same risk (x1)": 1.0, "three times the risk (x3)": 3.0}
rng = np.random.default_rng(0)  # seeded: the same subsets on every run


def slot_curves(data: dict, n: int, risk_scale: float) -> dict:
    """The equity curve of each ticker in a slot of CAPITAL / n, risking risk_scale x the budget / n."""
    config = replace(CONFIG, capital_brl=CAPITAL / n, max_risk_brl=CONFIG.max_risk_brl * risk_scale / n)
    return {t: slot_equity(data[t], config) for t in data}


def portfolio_of(curves: dict, members) -> dict:
    equity = combine_equity({t: curves[t] for t in members})
    summary = summarize_portfolio(equity, CAPITAL, cdi)
    held_index = compare_with_index(summary.cagr, summary.years, ibov, equity.index)
    return {**summary.__dict__, "ibov_cagr": held_index.cagr, "beats_ibov": held_index.beats}


def held_of(data: dict, members) -> dict:
    """Equal-weight buy and hold of the same tickers on the raw close (docs/ibov.md)."""
    held = summarize_portfolio(buy_and_hold_equity({t: data[t] for t in members}, CAPITAL), CAPITAL, cdi)
    return {"bh_cagr": held.cagr, "bh_max_equity_drawdown": held.max_equity_drawdown}


names = sorted(set(ibov_is) & set(ibov_oos))
by_score = scan_ibov["is_score_v1"].dropna().sort_values(ascending=False).index
by_score = [t for t in by_score if t in names]

rows = []
for n in PICK_SIZES:
    random_sets = [tuple(rng.choice(names, size=n, replace=False)) for _ in range(DRAWS)]
    for scale_name, scale in RISK_SCALES.items():
        curves = slot_curves(ibov_oos, n, scale)
        random_cagr = pd.Series([portfolio_of(curves, s)["cagr"] for s in random_sets])
        for label, members in (("best in-sample", by_score[:n]), ("worst in-sample (control)", by_score[-n:])):
            result = portfolio_of(curves, members)
            held = held_of(ibov_oos, members)
            rows.append({
                "selection": label, "tickers": n, "risk": scale_name,
                "oos cagr": result["cagr"], "oos equity dd": result["max_equity_drawdown"],
                "oos calmar": result["calmar"], "oos excess": result["excess"], "beats CDI": result["beats_cdi"],
                "IBOV cagr": result["ibov_cagr"], "beats IBOV": result["beats_ibov"],
                "B&H cagr": held["bh_cagr"], "beats B&H": result["cagr"] > held["bh_cagr"],
                "beats % of random": float((random_cagr < result["cagr"]).mean()),
            })
chosen = pd.DataFrame(rows).set_index(["selection", "tickers", "risk"]).sort_index()
print("best", PICK_SIZES[0], "in-sample:", by_score[: PICK_SIZES[0]])
chosen

**How to read it.** Compare the `best` and `worst` rows at each size: a ranking that means something puts `best` above `worst` and above 50% of random. Then ask whether the best row beats the CDI *and* the IBOV with a drawdown you could live with. A x3 row that beats the CDI by raising both return and drawdown has not found an edge, it has found leverage on one.

## 7. The ten most traded BDRs

No selection here: the universe is already the ten most traded candidates, so there is nothing to overfit by choosing. The table is the same fixed setting on each BDR, then the ten together as an equal-split portfolio out-of-sample.

In [ ]:
scan_bdr = scan_both(bdr_is, bdr_oos, MIN_CANDLES_BDR // 2, 100)
scan_bdr

In [ ]:
bdr_names = sorted(set(bdr_is) & set(bdr_oos))
rows = []
for scale_name, scale in RISK_SCALES.items():
    curves = slot_curves(bdr_oos, len(bdr_names), scale)
    result = portfolio_of(curves, bdr_names)
    held = held_of(bdr_oos, bdr_names)
    rows.append({
        "risk": scale_name, "tickers": len(bdr_names),
        "oos cagr": result["cagr"], "oos equity dd": result["max_equity_drawdown"],
        "oos calmar": result["calmar"], "oos excess": result["excess"], "beats CDI": result["beats_cdi"],
        "IBOV cagr": result["ibov_cagr"], "beats IBOV": result["beats_ibov"],
        "B&H cagr": held["bh_cagr"], "B&H equity dd": held["bh_max_equity_drawdown"],
        "beats B&H": result["cagr"] > held["bh_cagr"],
    })
pd.DataFrame(rows).set_index("risk")

**Caveats for the BDRs.** Ten correlated US mega-caps (many are tech), a short and dollar-driven history, and an out-of-sample window of only a few years: this is an anecdote, not evidence. The IBOV comparison is not like-for-like, and there is no US benchmark here.

## 8. What this notebook does not answer (yet)

- **Is the part-time exposure being judged fairly?** The CDI and the IBOV are fully invested every day, the strategy is not, so `excess` mixes edge with idle cash. A comparison over only the days in a trade, or a rule for what idle cash earns, needs an approved spec in `docs/` before any code.
- **Which signal wins when several fire together?** The independent slots have no shared cash, so a real account could not take every signal in a sell-off. That rule is a spec decision.
- **Does a pick made each year from past data keep working?** The single split in section 6b is one window. A walk-forward version needs a spec and tests too.

Write what you conclude from the tables above here, in your own words, before relying on any of them.